In [1]:
import pandas as pd
from protocols import utils, functions

%load_ext autoreload
%autoreload 2

In [2]:
# the path to the CRyPTIC v3.1.0 data tables (including the DST_MEASUREMENTS_+.pkl which has DST for the validation samples appended)
cryptic_tables_path = '/Users/fowler/Dropbox/files/cryptic/cryptic-release-three/cryptic-tables-v3.1.0/'

# whether to run the processes which take a long time
run_long_processes = False

# number of cores
n_cores = 6

drug_genes = {
    'BDQ': {"genes": ["Rv0678", "atpE", "pepQ"], "phylogenetic": []},
    'CFZ': {"genes": ["Rv0678", "atpE", "pepQ"], "phylogenetic": []},
    "AMI": {"genes": ["eis", "rrs"], "phylogenetic": []},
    "CAP": {"genes": ["rrs", "tlyA"], "phylogenetic": []},
    "DLM": {"genes": ["ddn"], "phylogenetic": []},
    "EMB": {"genes": ["embA", "embB"], "phylogenetic": []},
    "ETH": {"genes": ["ethA", "fabG1", "inhA"], "phylogenetic": []},
    "INH": {"genes": ["katG", "inhA", "ahpC", "fabG1"], "phylogenetic": []},
    "KAN": {"genes": ["eis", "rrs"], "phylogenetic": []},
    "LEV": {"genes": ["gyrA", "gyrB"], "phylogenetic": ['gyrA@S95T']},
    "LZD": {"genes": ["rplC"], "phylogenetic": []},
    "MXF": {"genes": ["gyrA", "gyrB"], "phylogenetic": ['gyrA@S95T']},
    "RIF": {"genes": ["rpoB"], "phylogenetic": []},
    "STM": {"genes": ["gid", "rpsL", "rrs"], "phylogenetic": []},
    'PZA': {"genes": ["pncA"], "phylogenetic": []},    
}

who_drugs = list(pd.read_csv('data/who2_drugs.csv').drug)

catalogues = {
        'WHOv1': 'catalogues/whov1/NC_000962.3_WHO-UCN-GTB-PCI-2021.7_v1.2_GARC1_RUS.csv',
        'WHOv2': 'catalogues/whov2/NC_000962.3_WHO-UCN-TB-2023.5_v2.0_GARC1_RFUS.csv',
        'CATv1': "catalogues/catomatic_v1.csv",
    }

In [3]:
phenotypes_RIF = functions.prep_phenotypes(
    'RIF',
    cryptic_tables_path+'DST_MEASUREMENTS_+.pkl',
    cryptic_tables_path+'GENOMES.parquet',
    cryptic_tables_path+'WGS_SAMPLES.parquet',
    'v3.0',
    validation=True
)
phenotypes_RIF.set_index('UNIQUEID', inplace=True)
phenotypes_RIF[:3]

,DRUG,PHENOTYPE,METHOD_MIC,METHOD_3
UNIQUEID,,,,
site.ENA.subj.ERR036186.lab.1.iso.1,RIF,S,NaN,NaN
site.ENA.subj.ERR036187.lab.1.iso.1,RIF,S,NaN,NaN
site.ENA.subj.ERR036188.lab.1.iso.1,RIF,S,NaN,NaN


In [4]:
drug_genes['RIF']['genes']

['rpoB']

In [5]:
mutations_RIF = functions.prep_mutations(
    'data/mutations-v3.1.0/', 
    drug_genes['RIF']['genes'], 
    version='v3.1.0', 
    mut_path=cryptic_tables_path+'MUTATIONS.parquet', 
    var_path=cryptic_tables_path+'VARIANTS.parquet',
    train=False
)
mutations_RIF.set_index('UNIQUEID', inplace=True)
mutations_RIF[:3]

,MUTATION,FRS
UNIQUEID,,
site.02.subj.0069.lab.22A019.iso.1,rpoB@S450L,1.0
site.02.subj.0069.lab.22A019.iso.1,rpoB@A1075A,1.0
site.ENA.subj.SAMEA2533644.lab.1.iso.1,rpoB@A1075A,1.0


In [6]:
all_RIF = phenotypes_RIF.join(mutations_RIF[mutations_RIF.FRS >= 0.1], how='left')
all_RIF.reset_index(inplace=True)
all_RIF[:3]

,UNIQUEID,DRUG,PHENOTYPE,METHOD_MIC,METHOD_3,MUTATION,FRS
0,site.ENA.subj.ERR036186.lab.1.iso.1,RIF,S,NaN,NaN,rpoB@A1075A,1.000000
1,site.ENA.subj.ERR036186.lab.1.iso.1,RIF,S,NaN,NaN,rpoB@G1010G,0.272727
2,site.ENA.subj.ERR036187.lab.1.iso.1,RIF,S,NaN,NaN,rpoB@G1010G,0.333333


In [7]:
results = []
for cat in ['WHOv1', 'WHOv2', 'CATv1']:
    for drug in ['RIF']:
        (i, a, b) = utils.piezo_predict(iso_df=all_RIF, drug=drug, catalogue_file=catalogues[cat], return_predictions=True)
        df2 = pd.DataFrame.from_dict({'id':i, 'prediction':b, 'label':a})
        df2['DRUG'] = drug
        df2['CATALOGUE'] = cat
        results.append(df2)
df_rif = pd.concat(results)
df_rif.set_index(['id','CATALOGUE'], inplace=True)
df_rif.rename(columns={'prediction':'RIF_pred', 'label':'RIF_label'}, inplace=True)
df_rif.drop(columns=['DRUG'], inplace=True)
df_rif[:3]

,,RIF_pred,RIF_label
id,CATALOGUE,,
site.ENA.subj.ERR036186.lab.1.iso.1,WHOv1,S,S
site.ENA.subj.ERR036187.lab.1.iso.1,WHOv1,S,S
site.ENA.subj.ERR036188.lab.1.iso.1,WHOv1,R,S


In [8]:
phenotypes_INH = functions.prep_phenotypes(
    'INH',
    cryptic_tables_path+'DST_MEASUREMENTS_+.pkl',
    cryptic_tables_path+'GENOMES.parquet',
    cryptic_tables_path+'WGS_SAMPLES.parquet',
    'v3.0',
    validation=True
)
phenotypes_INH.set_index('UNIQUEID', inplace=True)
phenotypes_INH[:3]

,DRUG,PHENOTYPE,METHOD_MIC,METHOD_3
UNIQUEID,,,,
site.ENA.subj.ERR036186.lab.1.iso.1,INH,S,NaN,NaN
site.ENA.subj.ERR036187.lab.1.iso.1,INH,S,NaN,NaN
site.ENA.subj.ERR036188.lab.1.iso.1,INH,R,NaN,NaN


In [9]:
mutations_INH = functions.prep_mutations(
    'data/mutations-v3.1.0/', 
    drug_genes['INH']['genes'], 
    version='v3.1.0', 
    mut_path=cryptic_tables_path+'MUTATIONS.parquet', 
    var_path=cryptic_tables_path+'VARIANTS.parquet',
    train=False
)
mutations_INH.set_index('UNIQUEID', inplace=True)
mutations_INH[:3]

,MUTATION,FRS
UNIQUEID,,
site.02.subj.0069.lab.22A019.iso.1,katG@S315T,1.0
site.02.subj.0069.lab.22A019.iso.1,katG@R463L,1.0
site.ENA.subj.SAMEA2533644.lab.1.iso.1,katG@R463L,1.0


In [10]:
all_INH = phenotypes_INH.join(mutations_INH[mutations_INH.FRS >= 0.1], how='left')
all_INH.reset_index(inplace=True)
all_INH[:3]

,UNIQUEID,DRUG,PHENOTYPE,METHOD_MIC,METHOD_3,MUTATION,FRS
0,site.ENA.subj.ERR036186.lab.1.iso.1,INH,S,NaN,NaN,fabG1@T4P,0.277778
1,site.ENA.subj.ERR036186.lab.1.iso.1,INH,S,NaN,NaN,katG@R463L,1.000000
2,site.ENA.subj.ERR036187.lab.1.iso.1,INH,S,NaN,NaN,NaN,NaN


In [11]:
results = []
for cat in ['WHOv1', 'WHOv2', 'CATv1']:
    for drug in ['INH']:
        (i, a, b) = utils.piezo_predict(iso_df=all_INH, drug=drug, catalogue_file=catalogues[cat], return_predictions=True)
        df2 = pd.DataFrame.from_dict({'id':i, 'prediction':b, 'label':a})
        df2['DRUG'] = drug
        df2['CATALOGUE'] = cat
        results.append(df2)
df_inh = pd.concat(results)
df_inh.set_index(['id','CATALOGUE'], inplace=True)
df_inh.rename(columns={'prediction':'INH_pred', 'label':'INH_label'}, inplace=True)
df_inh.drop(columns=['DRUG'], inplace=True)
df_inh[:3]

,,INH_pred,INH_label
id,CATALOGUE,,
site.ENA.subj.ERR036186.lab.1.iso.1,WHOv1,U,S
site.ENA.subj.ERR036187.lab.1.iso.1,WHOv1,S,S
site.ENA.subj.ERR036188.lab.1.iso.1,WHOv1,R,R


In [12]:
df = df_inh.join(df_rif, how='outer')
df.fillna('-',inplace=True)
df.to_csv('poster.csv')
df

INH_pred INH_label RIF_pred  \
id                                   CATALOGUE                               
site.ENA.subj.ERR036186.lab.1.iso.1  CATv1            U         S        S   
                                     WHOv1            U         S        S   
                                     WHOv2            U         S        S   
site.ENA.subj.ERR036187.lab.1.iso.1  CATv1            S         S        S   
                                     WHOv1            S         S        S   
...                                                 ...       ...      ...   
site.ENA.subj.SRR9851555.lab.1.iso.1 WHOv1            R         R        -   
                                     WHOv2            R         R        -   
site.ENA.subj.SRR9851564.lab.1.iso.1 CATv1            S         S        -   
                                     WHOv1            S         S        -   
                                     WHOv2            S         S        -   

                                               RIF_label  
id                                   CATALOGUE            
site.ENA.subj.ERR036186.lab.1.iso.1  CATv1             S  
                                     WHOv1             S  
                                     WHOv2             S  
site.ENA.subj.ERR036187.lab.1.iso.1  CATv1             S  
                                     WHOv1             S  
...                                                  ...  
site.ENA.subj.SRR9851555.lab.1.iso.1 WHOv1             -  
                                     WHOv2             -  
site.ENA.subj.SRR9851564.lab.1.iso.1 CATv1             -  
                                     WHOv1             -  
                                     WHOv2             -  

[14451 rows x 4 columns]

In [13]:
def process(row):
    mdr = False
    agree = False
    if row.INH_label=='-' or row.RIF_label=='-':
        has_both_label = False
    else:     
        has_both_label = True
    if row.INH_pred=='-' or row.RIF_pred=='-':
        has_both_pred = False
    else:     
        has_both_pred = True
    if row.INH_label == 'R' and row.RIF_label == 'R':
        mdr = True
        if row.INH_pred == 'R' and row.RIF_pred == 'R':
            agree = True 
    elif row.INH_label == 'S' and row.RIF_label == 'S':
        mdr = False
        if row.INH_pred in ['S', 'U'] and row.RIF_pred in ['S', 'U']:
            agree = True

    return pd.Series([has_both_pred, has_both_label, mdr,agree])

df[['HAS_PREDS','HAS_LABELS','MDR', 'AGREE']] = df.apply(process, axis=1)
df

INH_pred INH_label RIF_pred  \
id                                   CATALOGUE                               
site.ENA.subj.ERR036186.lab.1.iso.1  CATv1            U         S        S   
                                     WHOv1            U         S        S   
                                     WHOv2            U         S        S   
site.ENA.subj.ERR036187.lab.1.iso.1  CATv1            S         S        S   
                                     WHOv1            S         S        S   
...                                                 ...       ...      ...   
site.ENA.subj.SRR9851555.lab.1.iso.1 WHOv1            R         R        -   
                                     WHOv2            R         R        -   
site.ENA.subj.SRR9851564.lab.1.iso.1 CATv1            S         S        -   
                                     WHOv1            S         S        -   
                                     WHOv2            S         S        -   

                                               RIF_label  HAS_PREDS  \
id                                   CATALOGUE                        
site.ENA.subj.ERR036186.lab.1.iso.1  CATv1             S       True   
                                     WHOv1             S       True   
                                     WHOv2             S       True   
site.ENA.subj.ERR036187.lab.1.iso.1  CATv1             S       True   
                                     WHOv1             S       True   
...                                                  ...        ...   
site.ENA.subj.SRR9851555.lab.1.iso.1 WHOv1             -      False   
                                     WHOv2             -      False   
site.ENA.subj.SRR9851564.lab.1.iso.1 CATv1             -      False   
                                     WHOv1             -      False   
                                     WHOv2             -      False   

                                                HAS_LABELS    MDR  AGREE  
id                                   CATALOGUE                            
site.ENA.subj.ERR036186.lab.1.iso.1  CATv1            True  False   True  
                                     WHOv1            True  False   True  
                                     WHOv2            True  False   True  
site.ENA.subj.ERR036187.lab.1.iso.1  CATv1            True  False   True  
                                     WHOv1            True  False   True  
...                                                    ...    ...    ...  
site.ENA.subj.SRR9851555.lab.1.iso.1 WHOv1           False  False  False  
                                     WHOv2           False  False  False  
site.ENA.subj.SRR9851564.lab.1.iso.1 CATv1           False  False  False  
                                     WHOv1           False  False  False  
                                     WHOv2           False  False  False  

[14451 rows x 8 columns]

In [14]:
df[df.HAS_LABELS & df.HAS_PREDS].MDR.value_counts()/3

MDR
False    2468.0
True     2130.0
Name: count, dtype: float64

In [15]:
df.reset_index(inplace=True)

In [16]:
baa = df[(df.CATALOGUE=='WHOv1') & (df.HAS_PREDS) & (df.HAS_LABELS)]
pd.crosstab(baa.MDR,baa.AGREE,margins=True)

AGREE,False,True,All
MDR,,,
False,443,2025,2468
True,207,1923,2130
All,650,3948,4598


In [17]:
baa = df[(df.CATALOGUE=='WHOv2') & (df.HAS_PREDS) & (df.HAS_LABELS)]
pd.crosstab(baa.MDR,baa.AGREE,margins=True)

AGREE,False,True,All
MDR,,,
False,445,2023,2468
True,199,1931,2130
All,644,3954,4598


In [18]:
baa = df[(df.CATALOGUE=='CATv1') & (df.HAS_PREDS) & (df.HAS_LABELS)]
pd.crosstab(baa.MDR,baa.AGREE,margins=True)

AGREE,False,True,All
MDR,,,
False,426,2042,2468
True,218,1912,2130
All,644,3954,4598


In [19]:
baa[~baa.MDR & ~baa.AGREE]

,id,CATALOGUE,INH_pred,INH_label,RIF_pred,RIF_label,HAS_PREDS,HAS_LABELS,MDR,AGREE
6,site.ENA.subj.ERR036188.lab.1.iso.1,CATv1,R,R,R,S,True,True,False,False
36,site.ENA.subj.ERR036203.lab.1.iso.1,CATv1,R,R,S,S,True,True,False,False
123,site.ENA.subj.ERR036249.lab.1.iso.1,CATv1,R,R,S,S,True,True,False,False
144,site.ENA.subj.ERR037475.lab.1.iso.1,CATv1,R,R,S,S,True,True,False,False
180,site.ENA.subj.ERR037491.lab.1.iso.1,CATv1,R,R,S,S,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...
13005,site.ENA.subj.SRR6397642.lab.1.iso.1,CATv1,S,S,S,R,True,True,False,False
13035,site.ENA.subj.SRR6398132.lab.1.iso.1,CATv1,R,R,S,S,True,True,False,False
13137,site.ENA.subj.SRR6824317.lab.1.iso.1,CATv1,R,S,R,S,True,True,False,False
13722,site.ENA.subj.SRR6824512.lab.1.iso.1,CATv1,R,S,S,S,True,True,False,False
